# 🧠 The Mind of a Model
# Day 9: Model Deployment

In [1]:
!pip install gradio -q

print('✅ Gradio installed!')

✅ Gradio installed!


In [2]:
import pandas as pd
import numpy as np
import gradio as gr
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

print('✅ Libraries imported!')

✅ Libraries imported!


In [3]:
url = 'https://raw.githubusercontent.com/Aeshwa-Kachhadiya/The-mind-of-a-model/main/dataset/linkedin_post_performance.xlsx'

df = pd.read_excel(url)
print('✅ Dataset loaded!')
print(f'Shape: {df.shape}')

✅ Dataset loaded!
Shape: (300, 20)


In [4]:
binary_cols = ['has_image', 'has_video', 'has_carousel',
               'has_hashtags', 'has_question', 'has_emoji',
               'is_weekend']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['went_viral'] = df['went_viral'].map({'Yes': 1, 'No': 0})

le_topic = LabelEncoder()
le_day = LabelEncoder()

df['topic_category'] = le_topic.fit_transform(df['topic_category'])
df['posting_day'] = le_day.fit_transform(df['posting_day'])

numerical_cols = ['word_count', 'account_followers',
                  'account_age_months', 'posting_frequency',
                  'likes_first_hour', 'comments_first_hour',
                  'shares_first_hour', 'impressions_first_hour']
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

print('✅ Data prepared!')

✅ Data prepared!


/tmp/ipykernel_45470/4086931306.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [5]:
X = df.drop(columns=['went_viral'])
y = df['went_viral']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print('✅ Model trained!')
print(f'Feature columns: {list(X.columns)}')

✅ Model trained!
Feature columns: ['word_count', 'has_image', 'has_video', 'has_carousel', 'has_hashtags', 'hashtag_count', 'has_question', 'has_emoji', 'topic_category', 'posting_day', 'posting_hour', 'is_weekend', 'account_followers', 'account_age_months', 'posting_frequency', 'likes_first_hour', 'comments_first_hour', 'shares_first_hour', 'impressions_first_hour']


In [6]:
import pickle

with open('viral_predictor.pkl', 'wb') as f:
    pickle.dump(model, f)

print('✅ Model saved as viral_predictor.pkl')

✅ Model saved as viral_predictor.pkl


In [7]:
def predict_viral(word_count, has_image, has_video,
                  has_carousel, has_hashtags, hashtag_count,
                  has_question, has_emoji, topic_category,
                  posting_day, posting_hour, is_weekend,
                  account_followers, account_age_months,
                  posting_frequency, likes_first_hour,
                  comments_first_hour, shares_first_hour,
                  impressions_first_hour):

    # Encode inputs
    has_image = 1 if has_image == 'Yes' else 0
    has_video = 1 if has_video == 'Yes' else 0
    has_carousel = 1 if has_carousel == 'Yes' else 0
    has_hashtags = 1 if has_hashtags == 'Yes' else 0
    has_question = 1 if has_question == 'Yes' else 0
    has_emoji = 1 if has_emoji == 'Yes' else 0
    is_weekend = 1 if is_weekend == 'Yes' else 0

    # Encode topic and day
    try:
        topic_encoded = le_topic.transform([topic_category])[0]
    except:
        topic_encoded = 0

    try:
        day_encoded = le_day.transform([posting_day])[0]
    except:
        day_encoded = 0

    # Create input dataframe
    input_data = pd.DataFrame([{
        'word_count': word_count,
        'has_image': has_image,
        'has_video': has_video,
        'has_carousel': has_carousel,
        'has_hashtags': has_hashtags,
        'hashtag_count': hashtag_count,
        'has_question': has_question,
        'has_emoji': has_emoji,
        'topic_category': topic_encoded,
        'posting_day': day_encoded,
        'posting_hour': posting_hour,
        'is_weekend': is_weekend,
        'account_followers': account_followers,
        'account_age_months': account_age_months,
        'posting_frequency': posting_frequency,
        'likes_first_hour': likes_first_hour,
        'comments_first_hour': comments_first_hour,
        'shares_first_hour': shares_first_hour,
        'impressions_first_hour': impressions_first_hour
    }])

    # Make prediction
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0]

    if prediction == 1:
        return f'✅ This post will go VIRAL!\nConfidence: {probability[1]:.2%}'
    else:
        return f'❌ This post will NOT go viral.\nConfidence: {probability[0]:.2%}'

print('✅ Prediction function defined!')

✅ Prediction function defined!


In [8]:
import gradio as gr

# Custom CSS
css = """
.gradio-container {
    background: linear-gradient(135deg, #1a1a2e, #16213e, #0f3460);
    font-family: 'Georgia', serif;
}
.gr-button {
    background: linear-gradient(90deg, #6B8E4E, #4E6B8E) !important;
    border: none !important;
    color: white !important;
    font-size: 16px !important;
    padding: 12px 30px !important;
    border-radius: 25px !important;
}
.gr-button:hover {
    transform: scale(1.05) !important;
    box-shadow: 0 5px 15px rgba(107, 142, 78, 0.4) !important;
}
.gr-input, .gr-dropdown {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    color: white !important;
    border-radius: 10px !important;
}
.gr-panel {
    background: rgba(255,255,255,0.03) !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    border-radius: 15px !important;
}
"""

with gr.Blocks(css=css, theme=gr.themes.Base()) as app:

    # Header
    gr.Markdown("""
    # 🧠 The Mind of a Model
    ### Will your LinkedIn post go viral?
    *Fill in your post details and let the model decide.*
    ---
    """)

    with gr.Row():

        # Left Column — Post Content
        with gr.Column():
            gr.Markdown("### 📝 Post Content")

            word_count = gr.Number(
                label='Word Count',
                value=150)

            topic_category = gr.Dropdown(
                ['Tech', 'Career', 'Personal', 'ML', 'Motivation'],
                label='Topic Category',
                value='ML')

            with gr.Row():
                has_image = gr.Radio(
                    ['Yes', 'No'],
                    label='Has Image',
                    value='Yes')
                has_video = gr.Radio(
                    ['Yes', 'No'],
                    label='Has Video',
                    value='No')

            with gr.Row():
                has_carousel = gr.Radio(
                    ['Yes', 'No'],
                    label='Has Carousel',
                    value='No')
                has_hashtags = gr.Radio(
                    ['Yes', 'No'],
                    label='Has Hashtags',
                    value='Yes')

            with gr.Row():
                has_question = gr.Radio(
                    ['Yes', 'No'],
                    label='Has Question',
                    value='Yes')
                has_emoji = gr.Radio(
                    ['Yes', 'No'],
                    label='Has Emoji',
                    value='Yes')

            hashtag_count = gr.Slider(
                0, 10,
                value=3,
                label='Hashtag Count',
                step=1)

        # Middle Column — Timing + Account
        with gr.Column():
            gr.Markdown("### ⏰ Timing")

            posting_day = gr.Dropdown(
                ['Monday', 'Tuesday', 'Wednesday',
                 'Thursday', 'Friday', 'Saturday', 'Sunday'],
                label='Posting Day',
                value='Tuesday')

            posting_hour = gr.Slider(
                0, 23,
                value=8,
                label='Posting Hour (24hr)',
                step=1)

            is_weekend = gr.Radio(
                ['Yes', 'No'],
                label='Is Weekend',
                value='No')

            gr.Markdown("### 👤 Account Details")

            account_followers = gr.Number(
                label='Account Followers',
                value=1000)

            account_age_months = gr.Number(
                label='Account Age (months)',
                value=12)

            posting_frequency = gr.Slider(
                1, 15,
                value=3,
                label='Posts Per Week',
                step=1)

        # Right Column — Engagement + Result
        with gr.Column():
            gr.Markdown("### 📊 First Hour Engagement")

            likes_first_hour = gr.Number(
                label='Likes in First Hour',
                value=10)

            comments_first_hour = gr.Number(
                label='Comments in First Hour',
                value=5)

            shares_first_hour = gr.Number(
                label='Shares in First Hour',
                value=2)

            impressions_first_hour = gr.Number(
                label='Impressions in First Hour',
                value=500)

            gr.Markdown("### 🎯 Prediction")

            predict_btn = gr.Button(
                '🔮 Predict Virality',
                variant='primary')

            output = gr.Textbox(
                label='Result',
                lines=3,
                placeholder='Your prediction will appear here...')

    # Footer
    gr.Markdown("""
    ---
    *🧠 The Mind of a Model — A 10-day ML series by Aeshwa Kachhadiya*
    """)

    # Connect button to function
    predict_btn.click(
        fn=predict_viral,
        inputs=[
            word_count, has_image, has_video,
            has_carousel, has_hashtags, hashtag_count,
            has_question, has_emoji, topic_category,
            posting_day, posting_hour, is_weekend,
            account_followers, account_age_months,
            posting_frequency, likes_first_hour,
            comments_first_hour, shares_first_hour,
            impressions_first_hour
        ],
        outputs=output
    )

print('✅ Beautiful app built!')

/tmp/ipykernel_45470/447700547.py:34: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Base()) as app:
/tmp/ipykernel_45470/447700547.py:34: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Base()) as app:


✅ Beautiful app built!


In [9]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb9b2f407e39ee2dff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
